# Marketing Mix Model using Google Meridian

## 1. Install Meridian

In [1]:
# # Install meridian: from PyPI @ latest release
# !pip install git+https://github.com/google/meridian.git

# ## Note you need to install tesnorflow-probability 0.24.0 to avoid a version mismatch with the latest meridian release
# %pip install "tensorflow>=2.19,<2.21" "tf-keras>=2.18,<2.21" "tfp-nightly==0.26.0.dev20260130"

# # Install other dependencies
# !pip install tqdm

## 2. Load the Data from github

In [2]:
# prompt: git clone https://github.com/google/meridian.git
!git clone https://github.com/google/meridian.git

fatal: destination path 'meridian' already exists and is not an empty directory.


## 3. Load the necessary packages

In [3]:
import sys
!{sys.executable} -m pip install arviz
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_probability as tfp
import arviz as az

import IPython

from meridian import constants
from meridian.data import load
from meridian.data import test_utils
from meridian.model import model
from meridian.model import spec
from meridian.model import prior_distribution
from meridian.analysis import optimizer
from meridian.analysis import analyzer
from meridian.analysis import visualizer
from meridian.analysis import summarizer
from meridian.analysis import formatter

# check if GPU is available
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
print("Num CPUs Available: ", len(tf.config.experimental.list_physical_devices('CPU')))

Your runtime has 17.2 gigabytes of available RAM

Num GPUs Available:  0
Num CPUs Available:  1


In [4]:
# List down the varibales in the dataset and their corresponding column names
coord_to_columns = load.CoordToColumns(
    time='time',
    geo='geo',
    #controls=['GQV', 'Competitor_Sales'],
    population='population',
    kpi='conversions',
    revenue_per_kpi='revenue_per_conversion',
    media=[
        'Channel0_impression',
        'Channel1_impression',
        'Channel2_impression',
        'Channel3_impression',
        'Channel4_impression',
    ],
    media_spend=[
        'Channel0_spend',
        'Channel1_spend',
        'Channel2_spend',
        'Channel3_spend',
        'Channel4_spend',
    ],
    organic_media=['Organic_channel0_impression'],
    non_media_treatments=['Promo'],
)

In [5]:
correct_media_to_channel = {
    'Channel0_impression': 'Channel_0',
    'Channel1_impression': 'Channel_1',
    'Channel2_impression': 'Channel_2',
    'Channel3_impression': 'Channel_3',
    'Channel4_impression': 'Channel_4',
}
correct_media_spend_to_channel = {
    'Channel0_spend': 'Channel_0',
    'Channel1_spend': 'Channel_1',
    'Channel2_spend': 'Channel_2',
    'Channel3_spend': 'Channel_3',
    'Channel4_spend': 'Channel_4',
}

In [6]:
# Load the csv file data into a DataLoader object, which will be used to create the dataset for the model
loader = load.CsvDataLoader(
    csv_path="/Users/surajgurung/Library/CloudStorage/OneDrive-UniversityofFlorida/AI&FINtech/Marketing-Mix-Model/meridian/meridian/data/simulated_data/csv/geo_all_channels.csv",
    kpi_type='non_revenue',
    coord_to_columns=coord_to_columns,
    media_to_channel=correct_media_to_channel,
    media_spend_to_channel=correct_media_spend_to_channel,
)
data = loader.load()

In [7]:
# Define the prior distribution for ROI for each media channel

roi_mu = 0.2     # Mu for ROI prior for each media channel. (Mean)
roi_sigma = 0.9  # Sigma for ROI prior for each media channel. (Standard Deviation)
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(roi_mu, roi_sigma, name=constants.ROI_M)
)
model_spec = spec.ModelSpec(prior=prior)

mmm = model.Meridian(input_data=data, model_spec=model_spec)

2026-03-31 11:10:32.863631: I external/local_xla/xla/service/service.cc:163] XLA service 0x17fc623f0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-03-31 11:10:32.863649: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1774969832.879855 27681398 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [12]:
# Sample from the prior distribution to check if the model is specified correctly and to understand the implications of the chosen priors on the model parameters.
#%%time

mmm.sample_prior(500)
#It tells the function to generate 500 samples from the prior distribution.x

from tqdm.notebook import tqdm  # Import tqdm for Jupyter Notebook
import time

# ... (rest of your imports and code) ...

with tqdm(total=5*1000, desc="Training Progress") as pbar:  # Total iterations
    def update_progress(current_iteration, total_iterations):
        pbar.update(current_iteration)  # Update the progress bar

    # Remove progress_callback from the sample_posterior call
    mmm.sample_posterior(
        n_chains=3,
        n_adapt=500, #500 kept lower for testing purposes, can increase to 500 for better performance
        n_burnin=500, # 500 kept lower for testing purposes, can increase to 500 for better performance
        n_keep=1000, # 1000 kept lower for testing purposes, can increase to 1000 for better performance
        parallel_iterations=100,
        # progress_callback=update_progress  # Remove this line
    )
    # Manually update progress bar after sampling
    pbar.update(5 * 1000)

#n_chains: Markov Chain Monte Carlo (MCMC) indipendent chains
#n_adapt: number of initial samples used to tune the sampling algorithm for better performance
#n_burnin: number of initial samples from each chain that are discarded
#n_keep: number of samples to keep from each chain after the burn-in phase. These samples represent the posterior distribution of the model parameters and are used for inference.

Training Progress:   0%|          | 0/5000 [00:00<?, ?it/s]

2026-03-31 11:28:21.886028: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator mcmc_retry_init/assert_equal_1/Assert/AssertGuard/Assert
/Users/surajgurung/anaconda3/envs/AI_Fintech/lib/python3.10/site-packages/arviz/data/inference_data.py:157: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(
/Users/surajgurung/anaconda3/envs/AI_Fintech/lib/python3.10/site-packages/arviz/data/inference_data.py:1647: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(


R-hat close to 1.0 indicate convergence. R-hat < 1.2 indicates approximate convergence and is a reasonable threshold for many problems.

 R-hat (Gelman–Rubin statistic) measures: Whether your Markov chain Monte Carlo have converged to the same distribution.

 It measures between chain variance and within-chain variance.

 (Note: 1 independent simulation is 1 chain)

We are checking whether independent simulations converge to the same posterior distribution (our prior belief), indicating that the model has been properly learned.

## 1. Run the Model Diagnostics

In [15]:
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()

alt.LayerChart(...)

Each parameter in the plot corresponds to a specific aspect of your MMM model. Here's a general breakdown of the parameters commonly encountered in such models:

Alpha (
𝛼
α):

Represents the intercept of the model or the baseline contribution of the media channel (without interactions).
𝛼
𝑚
α
m
​
  and
𝛼
𝑜
𝑚
α
om
​
  might represent specific intercepts for different variables.

Beta (
𝛽
β):

Represents media elasticity. This measures the effect of media spend (e.g., TV, digital ads) on the outcome (e.g., sales).
𝛽
𝑔
𝑚
β
gm
​
 ,
𝛽
𝑔
𝑜
𝑚
β
gom
​
 ,
𝛽
𝑚
β
m
​
 , etc., might represent elasticities for different media channels or groups.

Gamma (
𝛾
γ):

Represents carryover effects or saturation parameters.
These parameters are often used in adstock transformations, capturing how the impact of media decays over time (e.g., long-term brand-building effects).

Eta (
𝜂
η):

Often represents the Hill transformation parameters for diminishing returns in MMM.
It controls the saturation level and steepness of the diminishing returns curve.

Sigma (
𝜎
σ):

Represents the standard deviation of the model residuals, indicating the uncertainty in your predictions.

Mu (
𝜇
μ):

Represents the mean of the response variable along with other contextual transformations.

Knot values:

Typically tied to spline-based models, representing flexible curves to capture nonlinear relationships (e.g., in spending vs. ROI).

Roi_m, Tau_g, Xi_c, Xi_n:

Likely represent channel-level ROI estimates (e.g., Return on Investment for media), time-dependent variables, or noise parameters depending on your specific model.


## 2. Assess the model's fit by comparing the expected revenue against the actual revenue.

In [11]:
mmm_summarizer = summarizer.Summarizer(mmm)
#save output

filepath = '/Users/surajgurung/Library/CloudStorage/OneDrive-UniversityofFlorida/AI&FINtech/Marketing-Mix-Model'
start_date = '2021-01-25'
end_date = '2024-01-15'
mmm_summarizer.output_model_results_summary('summary_output.html', filepath, start_date, end_date)
#preview 2 pager
IPython.display.HTML(filename='/Users/surajgurung/Library/CloudStorage/OneDrive-UniversityofFlorida/AI&FINtech/Marketing-Mix-Model/summary_output.html')


/Users/surajgurung/anaconda3/envs/AI_Fintech/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/Users/surajgurung/anaconda3/envs/AI_Fintech/lib/python3.10/site-packages/meridian/analysis/analyzer.py:3354: UserWarning: Effectiveness is not reported because it does not have a clear interpretation by time period.
  warnings.warn(


Dataset,R-squared,MAPE,wMAPE
All Data,0.90,2%,2%


Optimizing the budget

In [13]:
%%time
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()

CPU times: user 4min 55s, sys: 34.2 s, total: 5min 29s
Wall time: 2min 15s


In [14]:
filepath = '/Users/surajgurung/Library/CloudStorage/OneDrive-UniversityofFlorida/AI&FINtech/Marketing-Mix-Model'
optimization_results.output_optimization_summary('optimization_output.html', filepath)
IPython.display.HTML(filename='/Users/surajgurung/Library/CloudStorage/OneDrive-UniversityofFlorida/AI&FINtech/Marketing-Mix-Model/optimization_output.html')

Channel,Non-optimized spend,Optimized spend
Channel_3,40%,43%
Channel_0,18%,22%
Channel_4,22%,15%
Channel_1,14%,12%
Channel_2,6%,7%
